# Modelos de churn y segmentación de clientes

**Equipo:** Modelos · **Dataset:** `telco_churn_clean.csv`

Este notebook compara dos clasificadores para predecir `Churn` y crea segmentos con K-Means. Se conserva intacto el CSV entregado por Limpieza y EDA. La preparación supervisada vive dentro de un `Pipeline` para que el escalado y la codificación se ajusten solo con los datos de entrenamiento.

**Evaluación:** split estratificado 80/20, semilla fija; mostramos el baseline con umbral 0.50 y ajustamos un umbral para cada modelo solo con validación cruzada del train. Reportamos precision, recall, F1 y matriz de confusión para `Yes`. F1 de la clase `Yes` es el criterio principal porque equilibra precision y recall; precision y recall se reportan para hacer visible el intercambio.

> Las relaciones y probabilidades describen este dataset; no demuestran causas ni garantizan el comportamiento futuro.

## 1. Librerías y rutas

Si falta una librería en el entorno de VS Code, instálala en el entorno virtual del proyecto: `python3 -m pip install pandas numpy matplotlib seaborn scikit-learn jupyter`. En Mac, selecciona ese entorno como kernel del notebook.

In [ ]:
from pathlib import Path
import warnings
import json
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score, silhouette_score,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = next((root for root in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (root / ".git").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("No encontré la raíz del repositorio Git. Abre el notebook desde una copia del proyecto.")

PROJECT_DIR = REPO_ROOT
DATA_PATH = PROJECT_DIR / "data" / "processed" / "telco_churn_clean.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No encontré el CSV limpio en {DATA_PATH}. Actualiza la rama con los cambios del equipo de Limpieza y EDA."
    )

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos: {DATA_PATH}")
print(f"Salidas: {OUTPUT_DIR}")


## 2. Cargar y validar datos

`customerID` identifica al cliente y no es una señal predictiva. `Churn` es el objetivo; la convertimos a 1 para `Yes` y 0 para `No`. Las filas duplicadas por identificador deben revisarse antes de entrenar.

In [ ]:
df = pd.read_csv(DATA_PATH)

assert len(df) == 7043, f"Se esperaban 7043 filas; hay {len(df)}"
assert "customerID" in df.columns and "Churn" in df.columns
assert df["customerID"].is_unique, "customerID tiene duplicados"
assert df.isna().sum().sum() == 0, "El CSV contiene nulos"
assert set(df["Churn"].unique()) == {"Yes", "No"}

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
print(f"Dimensiones: {df.shape}; tasa churn: {y.mean():.1%}")
df["Churn"].value_counts().to_frame("clientes").assign(porcentaje=lambda x: x["clientes"] / len(df))


## 3. Separación train/test y preprocesamiento

El conjunto de prueba se reserva hasta la evaluación. One-hot encoding y escalado se ajustan dentro de cada pipeline con `X_train` solamente. Las categorías nuevas que aparezcan en test se ignoran de forma segura.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Proporción churn train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))


## 4. Entrenar y comparar clasificadores

Comparamos regresión logística como referencia interpretable y Random Forest como modelo no lineal. Ambos usan los mismos datos y preprocesamiento. El peso de clases se deja inicialmente sin ajustar para que las probabilidades y el umbral 0.50 sean una referencia clara; si el recall queda bajo, se puede ajustar el umbral usando solo validación de entrenamiento y volver a reportar el test una sola vez.

In [ ]:
models = {
    "Regresión logística": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_models = {}
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    rows.append({
        "modelo": name,
        "precision_yes": precision_score(y_test, pred, zero_division=0),
        "recall_yes": recall_score(y_test, pred, zero_division=0),
        "f1_yes": f1_score(y_test, pred, zero_division=0),
    })

metrics = pd.DataFrame(rows).set_index("modelo").sort_values("f1_yes", ascending=False)
metrics.round(3)


In [ ]:
for name, pipeline in fitted_models.items():
    pred = pipeline.predict(X_test)
    print(f"\n{name} — matriz de confusión @ umbral 0.50 (filas=real; columnas=predicción; orden No, Yes)")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))


In [ ]:
# Cada modelo se ajusta con su propio umbral, elegido solo con OOF del train; test queda reservado.
threshold_grid = np.linspace(0.05, 0.80, 151)
threshold_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tuned_thresholds = {}
tuned_rows = []
tuned_test_predictions = {}

for name, pipeline in fitted_models.items():
    train_oof_proba = cross_val_predict(
        pipeline, X_train, y_train, cv=threshold_cv, method="predict_proba", n_jobs=1
    )[:, 1]
    threshold = max(
        threshold_grid,
        key=lambda value: f1_score(y_train, train_oof_proba >= value, zero_division=0),
    )
    tuned_thresholds[name] = float(threshold)
    test_proba = pipeline.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)
    tuned_test_predictions[name] = test_pred
    tuned_rows.append({
        "modelo": name, "threshold": threshold,
        "precision_yes": precision_score(y_test, test_pred, zero_division=0),
        "recall_yes": recall_score(y_test, test_pred, zero_division=0),
        "f1_yes": f1_score(y_test, test_pred, zero_division=0),
    })

tuned_metrics = pd.DataFrame(tuned_rows).set_index("modelo").sort_values("f1_yes", ascending=False)
tuned_metrics.round(3)


In [ ]:
for name, pred in tuned_test_predictions.items():
    print(f"\n{name} — matriz de confusión @ umbral {tuned_thresholds[name]:.2f} (filas=real; columnas=predicción; orden No, Yes)")
    print(confusion_matrix(y_test, pred, labels=[0, 1]))


### Decisión del modelo supervisado

Reportamos la matriz de confusión, precision, recall y F1 para la clase positiva `Yes`. F1 es el criterio principal porque combina precision y recall; precision muestra qué parte de las alertas eran churn real, y recall qué parte de todos los churn reales detectó el modelo.

Primero se compara a ambos modelos con el umbral común 0.50. Después se elige el umbral de cada modelo maximizando F1 mediante validación cruzada dentro del train, y se evalúa una sola vez en el test reservado. En esta corrida, Regresión Logística eligió 0.33: F1 0.616, recall 0.719, precision 0.538; Random Forest eligió 0.49: F1 0.615, recall 0.725, precision 0.533. La diferencia es mínima. Con F1 como regla acordada, seleccionamos Regresión Logística a 0.33. El umbral bajo sube el recall respecto a 0.50, pero reduce precision; se eligió en train, no mirando el test.

## 5. K-Means: segmentación sin usar la respuesta

Para mantener segmentos interpretables, usamos las variables sugeridas por Limpieza: `tenure`, `MonthlyCharges` y `Contract`. Excluimos `Churn`, `customerID` y `TotalCharges`; de ese modo la etiqueta de churn no influye en la formación de clusters y la antigüedad no queda duplicada. Escalamos variables numéricas y codificamos contrato. Comparamos k=2…8 con silhouette; la métrica se usa junto a la interpretabilidad de los perfiles.

In [ ]:
cluster_features = ["tenure", "MonthlyCharges", "Contract"]
cluster_prep = ColumnTransformer([
    ("num", StandardScaler(), ["tenure", "MonthlyCharges"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Contract"]),
])
X_cluster = cluster_prep.fit_transform(df[cluster_features])

k_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_cluster)
    k_rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X_cluster, labels)})
k_results = pd.DataFrame(k_rows)
k_results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_results["k"], k_results["silhouette"], marker="o")
ax.set(xlabel="Número de clusters (k)", ylabel="Silhouette", title="Selección orientativa de k")
ax.set_xticks(k_results["k"])
plt.show()


Silhouette sube de 0.378 (`k=4`) a 0.431 (`k=8`) en el rango probado. Elegimos `k=8`: los perfiles distinguen subgrupos con contratos, antigüedad, cargo y churn diferentes. La solución requiere más campañas/seguimiento que k=4, así que su adopción depende de que el equipo pueda operar ocho segmentos. `Churn` se cruza después de formar los clusters; no se usa para entrenar K-Means.

In [ ]:
K_SELECTED = 8  # Mejor silhouette en el rango evaluado y perfiles diferenciados
cluster_pipeline = Pipeline([
    ("preprocess", cluster_prep),
    ("model", KMeans(n_clusters=K_SELECTED, random_state=RANDOM_STATE, n_init=20)),
])
# El pipeline recibe las columnas originales; el ColumnTransformer selecciona solo las tres acordadas.
segments = cluster_pipeline.fit_predict(X)

segmented = df[["customerID", "Churn", "tenure", "MonthlyCharges", "Contract", "InternetService", "PaymentMethod"]].copy()
segmented["segment"] = segments
SEGMENT_NAMES = {
    0: 'Mes a mes, fibra, antigüedad media-alta',
    1: 'Contratos largos, sin internet y cargo bajo',
    2: 'Contrato de dos años, antigüedad alta',
    3: 'Clientes nuevos, mes a mes y bajo cargo',
    4: 'Contrato anual, fibra y cargo alto',
    5: 'Mayormente dos años, sin internet y bajo cargo',
    6: 'Clientes recientes, mes a mes y fibra, alto riesgo',
    7: 'Contrato anual, DSL y antigüedad media',
}
segmented["segment_name"] = segmented["segment"].map(SEGMENT_NAMES)
segmented["churn_flag"] = y.values

segment_profile = segmented.groupby("segment").agg(
    clientes=("customerID", "size"),
    porcentaje=("customerID", lambda s: len(s) / len(segmented)),
    churn_rate=("churn_flag", "mean"),
    tenure_mediana=("tenure", "median"),
    cargo_mensual_mediana=("MonthlyCharges", "median"),
)
segment_profile["churn_rate"] *= 100
segment_profile["porcentaje"] *= 100
segment_profile.round(1)


In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\nDistribución de {col} dentro de cada segmento (%)")
    display(pd.crosstab(segmented["segment"], segmented[col], normalize="index").mul(100).round(1))


### Nombres y lectura de los segmentos

El notebook asigna etiquetas descriptivas al k seleccionado después de revisar tamaños y perfiles. Los nombres describen los perfiles observados, no causas del churn. Los IDs de K-Means son arbitrarios; revisa la correspondencia de nombres si cambian datos, semilla, k o versión de scikit-learn.

## 6. Exportar resultados para Power BI/API

El umbral se elige con el train mediante validación cruzada y F1; las métricas finales se calculan en el test reservado. El CSV por cliente usa scores out-of-fold de cinco folds, para que cada fila tenga una puntuación de un modelo que no vio esa fila. Incluye el ID técnico del cluster y un nombre descriptivo de la corrida actual. Los nombres de cluster deben revisarse si cambia la semilla, los datos o la versión de scikit-learn.

Para clientes nuevos, API debe entrenar y guardar un pipeline final con los datos acordados, el umbral seleccionado y este mismo preprocesamiento.

**Contrato de salida:** `telco_churn_model_output.csv` contiene `customerID`, `churn_probability`, `predicted_churn`, `segment` y `segment_name`. `telco_churn_dashboard_with_models.csv` contiene las columnas originales más estos cuatro campos. Los nombres de segmento y sus IDs se listan en `README_modelos.md`.

In [ ]:
BEST_MODEL_NAME = tuned_metrics.index[0]
SELECTED_THRESHOLD = tuned_thresholds[BEST_MODEL_NAME]
selected_model = fitted_models[BEST_MODEL_NAME]

# Probabilidades OOF: cada cliente queda fuera del fold que predice su registro.
final_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_probabilities = cross_val_predict(
    selected_model, X, y, cv=final_cv, method="predict_proba", n_jobs=1
)[:, 1]
oof_predictions = (oof_probabilities >= SELECTED_THRESHOLD).astype(int)

model_output = pd.DataFrame({
    "customerID": df["customerID"],
    "churn_probability": oof_probabilities,
    "predicted_churn": oof_predictions,
    "segment": segments,
    "segment_name": segmented["segment_name"],
})
model_output.to_csv(OUTPUT_DIR / "telco_churn_model_output.csv", index=False)
pbi_output = df.merge(model_output, on="customerID", validate="one_to_one")
pbi_output.to_csv(OUTPUT_DIR / "telco_churn_dashboard_with_models.csv", index=False)
print(f"Modelo seleccionado por F1 ajustado: {BEST_MODEL_NAME}; umbral={SELECTED_THRESHOLD:.2f}")
print("Guardado:", OUTPUT_DIR / "telco_churn_model_output.csv")
print("Guardado:", OUTPUT_DIR / "telco_churn_dashboard_with_models.csv")
print("Los artefactos de modelos se guardan después del cruce y del resumen de segmentos.")


## 7. Cruce de supervisado, K-Means y churn observado

Este análisis cruza por `customerID` el segmento, la probabilidad OOF y el churn real. Es una descripción posterior para evaluar cómo se concentra el riesgo; `Churn` no alimenta K-Means ni las predicciones. La tasa real del segmento y la probabilidad promedio ayudan a comparar el grupo observado con el score del clasificador.

La columna `parte_de_todos_los_churners` permite ver qué fracción de los clientes que abandonaron pertenecía a cada segmento.

In [ ]:
# Cruce post-hoc: K-Means + riesgo OOF + churn observado. Churn no intervino en K-Means ni en las predicciones OOF.
integrated = df[["customerID", "Churn"]].merge(
    model_output, on="customerID", validate="one_to_one"
)
integrated["churn_flag"] = (integrated["Churn"] == "Yes").astype(int)

segment_risk = integrated.groupby(["segment", "segment_name"]).agg(
    clientes=("customerID", "size"),
    tasa_del_total=("customerID", lambda s: len(s) / len(integrated) * 100),
    churn_real=("churn_flag", "mean"),
    probabilidad_media=("churn_probability", "mean"),
    churners=("churn_flag", "sum"),
).reset_index()
segment_risk["churn_real"] *= 100
segment_risk["probabilidad_media"] *= 100
segment_risk["parte_de_todos_los_churners"] = (
    segment_risk["churners"] / integrated["churn_flag"].sum() * 100
)
segment_risk.round(1)


In [ ]:
# Artefactos para API: ambos pipelines aceptan el DataFrame con campos originales.
segment_descriptions = {
    0: "100% mes a mes, 83% fibra; antigüedad mediana 43 meses.",
    1: "86% sin internet; contratos de uno o dos años; cargo mensual bajo.",
    2: "100% contrato de dos años; antigüedad mediana 69 meses; cargo mediano alto.",
    3: "Antigüedad mediana 5 meses; mes a mes; mayoría DSL y cargo bajo.",
    4: "100% contrato de un año; 78% fibra y cargo mensual alto.",
    5: "87% contrato de dos años; 78% sin internet y cargo mensual bajo.",
    6: "99% mes a mes; 85% fibra; antigüedad mediana 9 meses y churn más alto.",
    7: "96% contrato de un año; 82% DSL; antigüedad mediana 33 meses.",
}
segments_info = {
    "algorithm": "KMeans",
    "k": K_SELECTED,
    "columns": cluster_features,
    "random_state": RANDOM_STATE,
    "segments": [
        {
            "id": int(row.segment),
            "name": str(row.segment_name),
            "description": segment_descriptions[int(row.segment)],
            "customers": int(row.clientes),
            "churn_rate": round(float(row.churn_real) / 100, 4),
        }
        for row in segment_risk.itertuples(index=False)
    ],
}
with (MODEL_DIR / "segments_info.json").open("w", encoding="utf-8") as f:
    json.dump(segments_info, f, ensure_ascii=False, indent=2)
joblib.dump(cluster_pipeline, MODEL_DIR / "telco_customer_segments.joblib")

# Reentrenar el modelo final con todos los registros para predecir clientes nuevos.
# Las métricas de test anteriores no cambian; este artefacto es para despliegue.
deployment_model = selected_model.fit(X, y)
joblib.dump(deployment_model, MODEL_DIR / "telco_churn_logistic.joblib")
churn_model_info = {
    "model": "LogisticRegression",
    "artifact": "models/telco_churn_logistic.joblib",
    "threshold": SELECTED_THRESHOLD,
    "positive_class": "Yes",
    "input_columns": list(X.columns),
}
with (MODEL_DIR / "churn_model_info.json").open("w", encoding="utf-8") as f:
    json.dump(churn_model_info, f, ensure_ascii=False, indent=2)
print("Artefactos de API guardados en:", MODEL_DIR)


## 8. Resumen y limitaciones

Antes de compartir resultados, completa estas conclusiones con las métricas observadas y la decisión del equipo:

- Modelo supervisado elegido y justificación frente a las métricas de la clase `Yes`.
- Tamaño, perfil y tasa de churn de los ocho segmentos; usar las etiquetas definidas en `README_modelos.md`.
- Cruce entre segmento y probabilidad/ predicción de churn para identificar dónde concentrar revisión.
- Limitaciones: un solo dataset histórico, categorías y patrones específicos de esta empresa, costos de falsos positivos/falsos negativos no suministrados y asociación que no implica causalidad.
- Usar los nombres de archivo, columnas y segmentos definidos en `README_modelos.md` en la API y Power BI.

**Nota:** `predicted_churn` usa el umbral elegido en el train por validación cruzada y no debe considerarse recomendación definitiva de contacto hasta acordar capacidad y costos de retención. Las probabilidades del CSV de salida son out-of-fold; para servir predicciones nuevas, API debe entrenar y guardar un pipeline final con todo el conjunto de entrenamiento y respetar este mismo preprocesamiento.